In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, AddHs, RemoveHs, RWMol
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
import os
import re
import pandas as pd
import subprocess
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openpyxl import Workbook
import time
import shutil
import glob

In [ ]:
import os
os.environ['PATH'] += ':/path/to/example/static/Multiwfn_3.8_dev_bin_Linux'




os.environ['PATH'] = '/home/fwtop/apps/openmpi/bin:' + os.environ.get('PATH', '')

os.environ['LD_LIBRARY_PATH'] = '/home/fwtop/apps/openmpi/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:

import resource
resource.setrlimit(resource.RLIMIT_STACK, (resource.RLIM_INFINITY, resource.RLIM_INFINITY)) 

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:



import os

def create_empty_signal_file(file_name="pass1.sig", directory="."):
    
    
    file_path = os.path.join(directory, file_name)
    
    
    with open(file_path, 'w', encoding='utf-8') as f:
        
        pass
    
    
    print("Public message removed for release.")

In [ ]:

def generate_lowest_energy_conformer_openbabel(smiles: str, num_confs: int = 50, forcefield: str = "UFF"):
    
    
    obConversion = openbabel.OBConversion()
    obConversion.SetInFormat("smi")
    
    
    obmol = openbabel.OBMol()
    obConversion.ReadString(obmol, smiles)
    
    
    obmol.AddHydrogens()
    
    
    builder = openbabel.OBBuilder()
    builder.Build(obmol)
    
    
    cs = openbabel.OBConformerSearch()
    cs.Setup(obmol, num_confs, True)
    cs.Search()
    cs.GetConformers(obmol)
    
    
    forcefield = forcefield.upper()
    if forcefield == "MMFF94":
        ff = openbabel.OBForceField.FindForceField("mmff94")
    elif forcefield == "UFF":
        ff = openbabel.OBForceField.FindForceField("uff")
    else:
        raise ValueError("Public message removed for release.")
    
    lowest_energy = float('inf')
    lowest_conf_index = None
    nconfs = obmol.NumConformers()
    
    
    for i in range(nconfs):
        obmol.SetConformer(i)
        if not ff.Setup(obmol):
            print("Public message removed for release.")
            continue
        ff.ConjugateGradients(250, 1.0e-4)
        ff.GetCoordinates(obmol)
        energy = ff.Energy()
        if energy < lowest_energy:
            lowest_energy = energy
            lowest_conf_index = i

    
    if lowest_conf_index is None:
        return None, None, None
    
    atomic_number_to_symbol = {
        1: "H",    2: "He",   3: "Li",   4: "Be",   5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",
        11: "Na",  12: "Mg",  13: "Al",  14: "Si",  15: "P",   16: "S",   17: "Cl",  18: "Ar",  19: "K",   20: "Ca",
        21: "Sc",  22: "Ti",  23: "V",   24: "Cr",  25: "Mn",  26: "Fe",  27: "Co",  28: "Ni",  29: "Cu",  30: "Zn",
        31: "Ga",  32: "Ge",  33: "As",  34: "Se",  35: "Br",  36: "Kr",  37: "Rb",  38: "Sr",  39: "Y",   40: "Zr",
        41: "Nb",  42: "Mo",  43: "Tc",  44: "Ru",  45: "Rh",  46: "Pd",  47: "Ag",  48: "Cd",  49: "In",  50: "Sn",
        51: "Sb",  52: "Te",  53: "I",   54: "Xe",  55: "Cs",  56: "Ba",  57: "La",  58: "Ce",  59: "Pr",  60: "Nd",
        61: "Pm",  62: "Sm",  63: "Eu",  64: "Gd",  65: "Tb",  66: "Dy",  67: "Ho",  68: "Er",  69: "Tm",  70: "Yb",
        71: "Lu",  72: "Hf",  73: "Ta",  74: "W",   75: "Re",  76: "Os",  77: "Ir",  78: "Pt",  79: "Au",  80: "Hg",
        81: "Tl",  82: "Pb",  83: "Bi",  84: "Po",  85: "At",  86: "Rn",  87: "Fr",  88: "Ra",  89: "Ac",  90: "Th",
        91: "Pa",  92: "U",   93: "Np",  94: "Pu",  95: "Am",  96: "Cm",  97: "Bk",  98: "Cf",  99: "Es", 100: "Fm",
        101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg", 107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds",
        111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl", 115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }

    
    obmol.SetConformer(lowest_conf_index)
    coordinates = []
    
    for atom in pybel.ob.OBMolAtomIter(obmol):
        atomic_num = atom.GetAtomicNum()
        symbol = atomic_number_to_symbol.get(atomic_num, str(atomic_num))  
        x = atom.GetX()
        y = atom.GetY()
        z = atom.GetZ()
        coordinates.append((symbol, x, y, z))
    
    return lowest_conf_index, lowest_energy, coordinates

In [ ]:

def create_repeatunit_ORCA_inputfile(df, smiles, name, nproc=10, mem=2000):
    
    
    name_terminal_indices = {}
    name_non_terminal_indices = {}

    
    mol = Chem.MolFromSmiles(smiles)
    mol_with_h = AddHs(mol)

    
    terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']

    
    non_terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() != '*']

    
    rw_mol = RWMol(mol_with_h)

    
    for idx in terminal_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))  
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  

    
    rw_mol.UpdatePropertyCache(strict=False)

    
    new_mol = rw_mol.GetMol()

    
    new_mol_with_h = AddHs(new_mol)

    
    smiles_with_h = Chem.MolToSmiles(new_mol_with_h)

    
    lowest_conf_index, lowest_energy, coordinates_list = generate_lowest_energy_conformer_openbabel(smiles_with_h, num_confs=50, forcefield="UFF")
    
    
    if lowest_conf_index is None:
        print("Public message removed for release.")
        return
    
    
    coordinate_str = ""
    for atom_type, x, y, z in coordinates_list:
        coordinate_str += f"{atom_type} {x:.6f} {y:.6f} {z:.6f}\n"

    
    filename = name.replace(' ', '_') + '.inp'    

    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles_with_h)


      
    with open(filename, 'w') as f:
        
        f.write(f"! B3LYP D3 def2-TZVP def2/J RIJCOSX opt freq tightSCF noautostart miniprint\n")
        f.write(f"%maxcore     {mem}\n")
        f.write(f"%pal nprocs  {nproc} end\n")
        f.write(f"* xyz   {charge} {multiplicity}\n")
        f.write(coordinate_str)
        f.write(f"*")

    
    name_terminal_indices[name] = terminal_indices
    name_non_terminal_indices[name] = non_terminal_indices

    return name_terminal_indices, name_non_terminal_indices, charge

In [ ]:
def run_orca_gas_optfreq(file, success_dir, failure_dir):
    
    
    base_name = os.path.splitext(file)[0]
    output_file = base_name + '.out'
    opt_file = base_name + '.opt'
    trj_file = base_name + '_trj' + '.xyz'
    xyz_file = base_name + '.xyz'
    property_file = base_name + '.property' + '.txt'
    gbw_file = base_name + '.gbw'
    engrad_file = base_name + '.engrad'
    densitiesinfo_file = base_name + '.densitiesinfo'
    densities_file = base_name + '.densities'
    hess_file = base_name + '.hess'
    bibtex_file = base_name + '.bibtex'

    try:
        
        with open(output_file, 'w') as out:
            subprocess.run(['/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca', file], stdout=out, check=True)

        
        with open(output_file, 'r') as f:
            content = f.read()
            if 'ORCA TERMINATED NORMALLY' in content:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(success_dir, os.path.basename(f)))
                
                new_output_path = os.path.join(success_dir, os.path.basename(output_file))
                return new_output_path, True
            else:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(failure_dir, os.path.basename(f)))
                return None, False

    except subprocess.CalledProcessError as e:
        
        os.rename(file, os.path.join(failure_dir, os.path.basename(file)))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(failure_dir, os.path.basename(output_file)))
        print(f"Error running ORCA for file {file}: {str(e)}")
        print(traceback.format_exc())
        return None, False

In [ ]:

def calculate_repeatunit_RESP(filename):
    
    
    base_name = os.path.splitext(filename)[0]
    print(f"calculate_repeatunit_RESP Input: {base_name}")
    chgcons_file = base_name + '_chgcons' + '.txt'
    print(f"calculate_repeatunit_RESP chgfile Input: {chgcons_file}")
    
    commands = f'7\n18\n6\n1\n{chgcons_file}\n2\ny\n0\n0\nq'

    
    multiwfn_path = '/path/to/example/static/Multiwfn_3.8_dev_bin_Linux/Multiwfn'
    
    create_empty_signal_file(file_name="pass_2.sig")

    
    with open("Multiwfn_debug.txt", "w", encoding="utf-8") as debug_file:
        
        process = subprocess.Popen(
            [multiwfn_path, filename],
            stdin=subprocess.PIPE,   
            stdout=debug_file,       
            stderr=debug_file,       
            text=True                
        )
        
        
        create_empty_signal_file(file_name="pass_3.sig")
        
        
        process.communicate(commands)
    
    
    create_empty_signal_file(file_name="pass_4.sig")
    
    
    if process.returncode != 0:
        print("Public message removed for release.")
        
        

In [ ]:

import os
import subprocess
import shutil  

def convert_gbw_to_molden(path):
    
    
    for filename in os.listdir(path):
        
        if filename.endswith('.gbw'):
            
            base_filename = os.path.splitext(filename)[0]
            
            input_file = os.path.join(path, base_filename)
            
            command = [
                '/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca_2mkl',
                input_file,
                "-molden"
            ]

            try:
                
                subprocess.run(command, check=True)
                print(f"Converted {input_file} to {input_file}.molden.input")
                
                
                generated_file = os.path.join(path, base_filename + ".molden.input")
                
                current_dir = os.getcwd()
                
                dest_path = os.path.join(current_dir, base_filename + ".molden")
                
                shutil.copy(generated_file, dest_path)
                print(f"Copied {generated_file} to {dest_path}")
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {input_file}: {e}")

In [ ]:

def create_chgcons_files(non_terminal_indices, charge):
    
    for name, indices in non_terminal_indices.items():
        
        updated_indices = [index + 1 for index in indices]
        
        
        indices_str = ','.join(map(str, updated_indices))
        
        
        filename = f"{name}_chgcons.txt"
        
        
        with open(filename, 'w') as file:
            file.write(f"{indices_str} {charge}")

In [ ]:
def calculate(success_dir, failure_dir, MAX_TASKS=2):
    
    start_time = time.time()

    
    inp_files = [f for f in os.listdir('.') if os.path.isfile(f) and f.endswith('.inp')]

    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = [executor.submit(run_orca_gas_optfreq, file, success_dir, failure_dir) for file in inp_files] 
        
        results = [f.result() for f in futures]

    
    success_results = [result for result in results if result[1]]

    
    end_time = time.time()

    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")

In [ ]:

def create_repeatunit_chg(polymer_name):
    '''
    create_repeatunit_chg: BensanzuoCH3.molden.input
    calculate_repeatunit_RESP Input: BensanzuoCH3.molden
    calculate_repeatunit_RESP chgfile Input: BensanzuoCH3.molden_chgcons.txt
    '''
    for filename in os.listdir('.'):
        
        if filename.endswith('.molden'):
            
            for name in polymer_name:
                if filename.startswith(name):
                    
                    print(f"create_repeatunit_chg: {filename}")
                    calculate_repeatunit_RESP(filename)

                    
                    
                    found = False
                    for _ in range(300):
                        
                        if any(f.endswith('.chg') for f in os.listdir('.')):
                            print("Public message removed for release.")
                            found = True
                            break
                        
                        time.sleep(1)
                    if not found:
                        print("Public message removed for release.")

In [ ]:

def read_excel_and_create_lists(filepath):
    
    df = pd.read_excel(filepath)
    
    polymers = df[df['is polymer'] == True]['Name'].astype(str).tolist()
    small_molecules = df[df['is polymer'] == False]['Name'].astype(str).tolist()
    return df, polymers, small_molecules

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:

def copy_files_for_polymers(source_directory, destination_directory, polymers_list):
    
    os.makedirs(destination_directory, exist_ok=True)
    
    unique_filenames = get_unique_filenames(source_directory)
    
    not_found_polymers = []
    found_polymers = []
    
    
    for polymer in polymers_list:
        if polymer in unique_filenames:
            found_polymers.append(polymer)
            for extension in ['.inp', '.out', '.opt', '.xyz', '.txt', '.gbw', '.engrad', '.densitiesinfo', '.densities', '.hess', '.bibtex']:
                source_file = os.path.join(source_directory, polymer + extension)
                destination_file = os.path.join(destination_directory, polymer + extension)
                if os.path.exists(source_file):
                    try:
                        shutil.copyfile(source_file, destination_file)
                    except Exception as e:
                        print(f"Error copying {source_file}: {e}")
                else:
                    print(f"File not found: {source_file}")
            
            polymer_chgcons = f"{polymer}_chgcons.txt"
            
            source_file = os.path.join(source_directory, polymer_chgcons)
            destination_file = polymer_chgcons
            if os.path.exists(source_file):
                try:
                    shutil.copyfile(source_file, destination_file)
                except Exception as e:
                    print(f"Error copying {source_file}: {e}")
            else:
                print(f"File not found: {source_file}")
        else:
            not_found_polymers.append(polymer)
            
    
    print("Found polymers:", found_polymers)
    
    print("Polymers not found:", not_found_polymers)
    return not_found_polymers, found_polymers

In [ ]:

import os
import shutil

def copy_newly_calculated_files(RESPpolymer_database_path, success_directory, not_found_polymers):
    
    
    os.makedirs(RESPpolymer_database_path, exist_ok=True)

    
    success_filenames = get_unique_filenames(success_directory)

    
    extensions = [
        '.inp', '.out', '.opt', '.xyz', '.txt', '.gbw',
        '.engrad', '.densitiesinfo', '.densities', '.hess', '.bibtex'
    ]

    
    for polymer in not_found_polymers:
        
        if polymer in success_filenames:
            
            for ext in extensions:
                src = os.path.join(success_directory, polymer + ext)
                dst = os.path.join(RESPpolymer_database_path, polymer + ext)
                if os.path.isfile(src):
                    try:
                        shutil.copyfile(src, dst)
                        print("Public message removed for release.")
                    except Exception as e:
                        print("Public message removed for release.")
                else:
                    print("Public message removed for release.")

            
            chgcons_name = f"{polymer}_chgcons.txt"
            src_chg = chgcons_name
            dst_chg = os.path.join(RESPpolymer_database_path, chgcons_name)
            if os.path.isfile(src_chg):
                try:
                    shutil.copyfile(src_chg, dst_chg)
                    print("Public message removed for release.")
                except Exception as e:
                    print("Public message removed for release.")
            else:
                print("Public message removed for release.")
        else:
            
            print("Public message removed for release.")

In [ ]:

def create_input_files_for_missing_polymers(df, missing_polymers):
    for name in missing_polymers:
        smiles = df[df['Name'] == name]['SMILES'].iloc[0]
        
        terminal_indices, non_terminal_indices, repeatunit_charge = create_repeatunit_ORCA_inputfile(df, smiles, name)
        
        create_chgcons_files(non_terminal_indices, repeatunit_charge)
        print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")

In [ ]:

def check_and_create_orca_database(orca_database_path='/data/ORCA_database'):
    
    home_directory = '/data'
    
    optfreq_orca_database_path = os.path.join(home_directory, 'ORCA_database/opt+freq')
    RESPpolymer_database_path = os.path.join(home_directory, 'ORCA_database/RESPpolymer')
    
    
    if not os.path.exists(orca_database_path):
        
        os.makedirs(orca_database_path)
        os.makedirs(optfreq_orca_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(optfreq_orca_database_path) and os.path.exists(RESPpolymer_database_path):
        os.makedirs(optfreq_orca_database_path)
        print("Public message removed for release.")
        
    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_orca_database_path):
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_orca_database_path):
        os.makedirs(optfreq_orca_database_path)
        os.makedirs(RESPpolymer_database_path)
        print("Public message removed for release.")
        
    else:
        
        print("Public message removed for release.")
    return orca_database_path, optfreq_orca_database_path, RESPpolymer_database_path

In [ ]:

os.makedirs('Gaussian/RESPpolymer/success', exist_ok=True)
os.makedirs('Gaussian/RESPpolymer/failure', exist_ok=True)


base_dir = 'Gaussian/RESPpolymer'
success_dir = os.path.join(base_dir, 'success')
failure_dir = os.path.join(base_dir, 'failure')


Gaussian_path = "Gaussian/RESPpolymer/success"


gaussian_database_path, optfreq_gaussian_database_path, RESPpolymer_database_path = check_and_create_orca_database()


df, polymers, molecules = read_excel_and_create_lists('System.xlsx')


not_found_polymers, found_polymers = copy_files_for_polymers(
    RESPpolymer_database_path,
    success_dir,
    polymers
)
print(f"not_found_polymers: {not_found_polymers}")
print(f"found_polymers: {found_polymers}")

total_polymer_list = not_found_polymers+found_polymers


create_input_files_for_missing_polymers(df, not_found_polymers)



calculate(success_dir, failure_dir, MAX_TASKS=1)  


copy_newly_calculated_files(RESPpolymer_database_path, success_dir, not_found_polymers)


convert_gbw_to_molden(Gaussian_path)
create_empty_signal_file(file_name="pass_1.sig")

create_repeatunit_chg(total_polymer_list)
create_empty_signal_file(file_name="pass_5.sig")
